# Three-country falsification: $\mathcal S_c \approx \mathcal S_G \oplus \mathcal S_{L,c}$

**Run directory:** `week5/runs/three_country_falsification_20261005_185246`

Full report + analysis figures (no recompute required).

- Report: `../runs/three_country_falsification_20261005_185246/REPORT.md`
- Config: `../runs/three_country_falsification_20261005_185246/config.json`
- Artifacts: `../runs/three_country_falsification_20261005_185246/artifacts/`


# Three-country falsification — Global ⊕ Local task subspace

**Run:** `three_country_falsification_20261005_185246`
**Hypothesis:** `S_c ≈ S_G ⊕ S_L,c`
**Prior bases:** `/home/kv133/Alpha-Earth-Embeddings-Research/week5/runs/global_local_subspace_20261005_173516`
**Seeds:** [0, 1, 2, 3, 4] · **max_per_class:** 5000 · **r\*:** 1

## Success criteria

**Overall: `partial_mixed`**

| # | Criterion | Result | Evidence |
|---|---|---|---|
| 1 | Significant shared task-subspace structure | FAIL | λ1=0.986 (null95=0.987), λ2=0.982 (null95=0.982) |
| 2 | $S_G$ retains strong LOCO cross-country signal | PASS | LOCO global F1=0.657 vs lr64=0.510 |
| 3 | $S_{L,c}$ adds within-country information | PASS | within SG⊕SL=0.733 > SG=0.710 |
| 4 | Matched local beats mismatched | PASS | mean diagonal advantage=0.009; targets with CI>0: 2/3 |

## Interpretation guide

- **all_four** → enough evidence to justify scaling globally
- **partial_1_2** → transferable common subspace, weak local structure
- **partial_no_country_specific** → global+residual helps, but residual is not country-specific
- **contradicts** → shared supervised geometry not supported

## Representation vs OOD

- Descriptive $S_G$ (all three countries): spectrum, within-country, swap structure
- LOCO $S_G^{(-c)}$: all cross-country transfer claims

## Diagonal advantage by target

| target | r | matched F1 | mismatched F1 | advantage | 95% CI |
|---|---:|---:|---:|---:|---|
| USA | 1 | 0.837 | 0.827 | 0.011 | [0.010, 0.012] |
| JPN | 1 | 0.719 | 0.719 | 0.000 | [-0.000, 0.001] |
| IND | 1 | 0.642 | 0.627 | 0.015 | [0.011, 0.018] |

## Rank phenomenon (exploratory)

- Late-k (k≥10) mean ΔF1 within=+0.0005, ΔF1 OOD=-0.0043
- Early-k (2–5) mean ΔGlobalOverlap=+0.5722

## Figures
- `01_avgproj_spectrum_vs_nulls.png`
- `02_sg_sl_prediction_heatmap.png`
- `03_local_swap_matrix.png`
- `04_rank_f1_globaloverlap.png`
- `05_success_criteria_dashboard.png`

## Reproducibility
- Config: `config.json`
- r*: `artifacts/r_star.json`
- Artifacts: `artifacts/*.csv`
- Notebook: `week5/notebooks/04_three_country_falsification.ipynb`



## Figures


### 1. Average-projection spectrum vs nulls

![1. Average-projection spectrum vs nulls](../runs/three_country_falsification_20261005_185246/figures/01_avgproj_spectrum_vs_nulls.png)

`01_avgproj_spectrum_vs_nulls.png`


### 2. SG / SL prediction table

![2. SG / SL prediction table](../runs/three_country_falsification_20261005_185246/figures/02_sg_sl_prediction_heatmap.png)

`02_sg_sl_prediction_heatmap.png`


### 3. Local-component swap matrix

![3. Local-component swap matrix](../runs/three_country_falsification_20261005_185246/figures/03_local_swap_matrix.png)

`03_local_swap_matrix.png`


### 4. Rank phenomenon vs GlobalOverlap

![4. Rank phenomenon vs GlobalOverlap](../runs/three_country_falsification_20261005_185246/figures/04_rank_f1_globaloverlap.png)

`04_rank_f1_globaloverlap.png`


### 5. Success-criteria dashboard

![5. Success-criteria dashboard](../runs/three_country_falsification_20261005_185246/figures/05_success_criteria_dashboard.png)

`05_success_criteria_dashboard.png`


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

RUN = Path(r"week5/runs/three_country_falsification_20261005_185246")
display(Markdown("### r_star"))
display(json.loads((RUN / "artifacts" / "r_star.json").read_text()))
display(Markdown("### success criteria"))
display(json.loads((RUN / "artifacts" / "success_criteria.json").read_text()))
pred = pd.read_csv(RUN / "artifacts" / "sg_sl_prediction_table.csv")
rs = json.loads((RUN / "artifacts" / "r_star.json").read_text())["r_star"]
summary = (
    pred[pred.r == rs]
    .groupby(["protocol", "mode", "representation"])["m_macro_f1"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
display(Markdown("### SG/SL F1 summary at r*"))
display(summary)
display(Markdown("### Local swap diagonal advantage"))
display(pd.read_csv(RUN / "artifacts" / "local_swap_summary.csv"))
rank = pd.read_csv(RUN / "artifacts" / "rank_overlap.csv").dropna(subset=["delta_f1_ood", "delta_global_overlap"])
if len(rank):
    corr = rank["delta_f1_ood"].corr(rank["delta_global_overlap"])
    display(Markdown(f"### corr(ΔF1_OOD, ΔGlobalOverlap) = **{corr:.3f}**"))
display(rank.head(12))
